# dependencies

In [1]:
!pip -q install sentence-transformers chromadb


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


# mounting on drive, loading the raw data, and selectin the top 30 stock based on the number of articles

In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

import os, hashlib, numpy as np, pandas as pd, chromadb
from sentence_transformers import SentenceTransformer

# BASE = "/content/drive/MyDrive/ie624_project/data"   # if on drive 
BASE = "data"        # locally

full = pd.read_parquet(f"{BASE}/raw/combined.parquet")
# dropping EA and WBA, as they were delisted
ranked = [t for t in full["Stock_symbol"].value_counts().index if t not in ("EA", "WBA")]
keep = ranked[:30]
d = full[full["Stock_symbol"].isin(keep)].copy()
d = d.drop(columns=["Unnamed: 0", "dt", "Author", "Publisher",
                    "Lsa_summary", "Luhn_summary"], errors="ignore")
print(d.shape, d.columns.tolist())
print(keep)

(220881, 7) ['Date', 'Article_title', 'Stock_symbol', 'Url', 'Article', 'Textrank_summary', 'Lexrank_summary']
['GILD', 'NVDA', 'INTC', 'TSLA', 'GOOG', 'MU', 'AAPL', 'AMD', 'QCOM', 'CMCSA', 'MSFT', 'SBUX', 'EBAY', 'PEP', 'TXN', 'COST', 'ADBE', 'BIIB', 'PANW', 'PYPL', 'CSX', 'VRTX', 'AMGN', 'AMZN', 'SIRI', 'DLTR', 'TMUS', 'AMAT', 'ADI', 'AVGO']


## creating the docs document

In [2]:
ts = pd.to_datetime(d["Date"], utc=True, errors="coerce")
x, t = d[ts.notna()], ts[ts.notna()]

title = x["Article_title"].fillna("").str.strip()
art = x["Article"].fillna("")
lex = x["Lexrank_summary"].fillna("").str.strip()
pt = ((t - pd.Timestamp("1970-01-01", tz="UTC")).dt.total_seconds()).astype("int64")

is_long = (title + ". " + art).str.len() > 5000
use_sum = is_long & (lex.str.len() > 0)

docs = pd.DataFrame({
    "ticker": x["Stock_symbol"],
    "title": title,
    "text": (title + ". " + art).str.slice(0, 3000),
    "embed_text": (title + ". " + lex).where(use_sum, title + ". " + art),
    "embed_source": np.where(use_sum, "summary", "article"),
    "publish_time": pt,
    "date_only": t.dt.strftime("%H:%M:%S") == "00:00:00",
    "source_type": "news",
    "source_name": "nasdaq",
    "url": x["Url"],
    "textrank_summary": x["Textrank_summary"],
    "lexrank_summary": x["Lexrank_summary"],
})
docs = docs[docs["text"].str.len() > 40]
docs = docs.drop_duplicates(subset=["ticker", "title", "publish_time"]).reset_index(drop=True)
docs["doc_id"] = [hashlib.sha1(f"{a}|{b}|{c}".encode()).hexdigest()[:16]
                  for a, b, c in zip(docs.ticker, docs.publish_time, docs.title)]
print(docs.shape, docs.embed_source.value_counts().to_dict())

(209262, 13) {'article': 148279, 'summary': 60983}


In [5]:
docs.to_parquet(f"{BASE}/raw/cleaned_data.parquet", index=False)

# embedding all the articles and storing it in chroma

In [7]:
"""
Training if not present
"""

model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
model.max_seq_length = 512
os.makedirs(f"{BASE}/emb_shards", exist_ok=True)

col = chromadb.PersistentClient(path="/content/chroma_full").get_or_create_collection(
    "news", metadata={"hnsw:space": "cosine"})

def add_to_chroma(p, e):
    for i in range(0, len(p), 512):
        b, eb = p.iloc[i:i+512], e[i:i+512]
        col.upsert(
            ids=b.doc_id.tolist(), embeddings=eb.tolist(), documents=b.text.tolist(),
            metadatas=[{"ticker": r.ticker, "publish_time": int(r.publish_time),
                        "title": r.title, "url": r.url or "",
                        "embed_source": r.embed_source,
                        "lexrank_summary": r.lexrank_summary or ""}
                       for r in b.itertuples()])

done = set(col.get(include=[])["ids"])
S = 10_000
for k, s in enumerate(range(0, len(docs), S)):
    part = docs.iloc[s:s+S]
    ids = part.doc_id.tolist()
    f = f"{BASE}/emb_shards/shard_{k:03d}.npz"
    if os.path.exists(f):
        z = np.load(f, allow_pickle=True)          # your own file, safe to load
        assert list(z["ids"]) == ids, "docs order changed!"
        emb = z["emb"]
    else:
        emb = model.encode(part.embed_text.tolist(), batch_size=128,
                           normalize_embeddings=True, show_progress_bar=False).astype("float32")
        np.savez(f, ids=np.array(ids, dtype="U16"), emb=emb)   # plain strings, no pickle needed
    new = ~part.doc_id.isin(done).values
    add_to_chroma(part[new], emb[new])
    done.update(part.doc_id[new])
    print(f"shard {k} done | chroma count {col.count()}", flush=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

shard 0 done | chroma count 87168
shard 1 done | chroma count 87168
shard 2 done | chroma count 87168
shard 3 done | chroma count 87168
shard 4 done | chroma count 87168
shard 5 done | chroma count 87168
shard 6 done | chroma count 87168
shard 7 done | chroma count 87168
shard 8 done | chroma count 90000
shard 9 done | chroma count 100000
shard 10 done | chroma count 110000
shard 11 done | chroma count 120000
shard 12 done | chroma count 130000
shard 13 done | chroma count 140000
shard 14 done | chroma count 150000
shard 15 done | chroma count 160000
shard 16 done | chroma count 170000
shard 17 done | chroma count 180000
shard 18 done | chroma count 190000
shard 19 done | chroma count 200000
shard 20 done | chroma count 209262


## loading the chroma (already trained)

In [4]:
import shutil
if not os.path.exists("/content/chroma_full"):
    shutil.copytree(f"{BASE}/chroma_full", "/content/chroma_full")   # local copy is safer than reading from Drive

col = chromadb.PersistentClient(path="/content/chroma_full").get_collection("news")
model = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
print(col.count(), len(docs))      # the two numbers should match

KeyboardInterrupt: 

In [ ]:
!mkdir -p {BASE}/chroma_full
!cp -r /content/chroma_full/. {BASE}/chroma_full/
print(col.count(), len(docs))
print(col.get(limit=3, include=["metadatas"])["metadatas"])

209262 209262
[{'url': 'https://www.nasdaq.com/articles/my-6-largest-portfolio-holdings-heading-into-2024-and-the-important-investing-lesson-i', 'title': 'My 6 Largest Portfolio Holdings Heading Into 2024 -- and the Important Investing Lesson I Learned From Each One', 'ticker': 'AAPL', 'publish_time': 1702764000, 'lexrank_summary': '3: Apple There\'s little question that Apple (NASDAQ: AAPL) has become one of the most successful companies in history. 6: Nvidia Every investor has one -- the "stock that got away." Should you invest $1,000 in Nvidia right now?', 'embed_source': 'summary'}, {'publish_time': 1702764000, 'embed_source': 'summary', 'lexrank_summary': "Brokerage Recommendation Trends for AAPL Let's take a look at what these Wall Street heavyweights have to say about Apple (AAPL) before we discuss the reliability of brokerage recommendations and how to use them to your advantage. Is AAPL a Good Investment?", 'url': 'https://www.nasdaq.com/articles/brokers-suggest-investing-in-a

## finbert model on all article to find the sentiment of all articles related to stock

In [ ]:
import glob, torch
from transformers import pipeline

clf = pipeline("text-classification", model="ProsusAI/finbert", top_k=None,
               device=0, torch_dtype=torch.float16)

out_dir = f"{BASE}/finbert_parts"; os.makedirs(out_dir, exist_ok=True)
P = 10_000
for k, s in enumerate(range(0, len(docs), P)):
    f = f"{out_dir}/part_{k:03d}.parquet"
    if os.path.exists(f):
        continue
    part = docs.iloc[s:s+P]
    res = clf(part.embed_text.tolist(), batch_size=64, truncation=True, max_length=512)
    rows = []
    for doc_id, r in zip(part.doc_id, res):
        p = {x["label"].lower(): x["score"] for x in r}
        rows.append({"doc_id": doc_id, "pos": p["positive"], "neg": p["negative"],
                     "neu": p["neutral"], "sentiment": p["positive"] - p["negative"]})
    pd.DataFrame(rows).to_parquet(f, index=False)
    print("saved part", k, flush=True)

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

saved part 0
saved part 1
saved part 2


In [ ]:
fb = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(f"{out_dir}/part_*.parquet"))],
               ignore_index=True)
fb.to_parquet(f"{BASE}/finbert_results.parquet", index=False)
print(fb.shape, len(docs), fb.doc_id.is_unique)
print(fb.sentiment.describe())

## loading the finbert processed dataset

# Running a query on chromadb

In [9]:
import math, random
import pandas as pd

DAY = 86400
ts = lambda d: int(pd.Timestamp(d, tz="UTC").timestamp())

def month_edges(s_ts, e_ts):
    s = pd.Timestamp(s_ts, unit="s", tz="UTC")
    e = pd.Timestamp(e_ts, unit="s", tz="UTC")
    edges, cur = [s], s.normalize().replace(day=1) + pd.offsets.MonthBegin(1)
    while cur <= e:
        edges.append(cur)
        cur += pd.offsets.MonthBegin(1)
    edges.append(e + pd.Timedelta(seconds=1))
    return [int(x.timestamp()) for x in edges]

def run_query(qe, ticker, lo, hi, n):
    conds = []
    if lo is not None: conds.append({"publish_time": {"$gte": lo}})
    if hi is not None: conds.append({"publish_time": {"$lte": hi}})
    if ticker: conds.append({"ticker": ticker})
    where = None if not conds else conds[0] if len(conds) == 1 else {"$and": conds}
    r = col.query(query_embeddings=qe, n_results=n, where=where,
                  include=["metadatas", "documents", "distances"])
    return [{"doc_id": i, **m, "text": t, "sim": 1 - d}
            for i, m, t, d in zip(r["ids"][0], r["metadatas"][0],
                                  r["documents"][0], r["distances"][0])]

def retrieve(query, ticker, start, end, as_of,
             k_per_bucket=3, n_candidates=10, k_total=15,
             use_asof=True, use_window=True, use_decay=True, use_buckets=True,
             alpha=0.7, half_life=30):
    # Plain-RAG baseline: use_window=False, use_asof=False, use_decay=False, use_buckets=False
    qe = model.encode([query], normalize_embeddings=True).tolist()
    as_of_ts = ts(as_of) + DAY - 1              # as_of = end of that UTC day, inclusive
    lo, hi = ts(start), ts(end) + DAY - 1
    if use_asof:
        hi = min(hi, as_of_ts)                  # window can never pass the cutoff
    if not use_window:
        lo = None
        hi = as_of_ts if use_asof else None     # keep the cutoff if as-of is on
        use_buckets = False

    if use_buckets:
        edges = month_edges(lo, hi)
        rows = []
        for i in range(len(edges) - 1):
            rows += [dict(r, bucket=i)
                     for r in run_query(qe, ticker, edges[i], edges[i + 1] - 1, n_candidates)]
    else:
        rows = [dict(r, bucket=0) for r in run_query(qe, ticker, lo, hi, 50)]

    df = pd.DataFrame(rows)
    if df.empty:
        return df
    age = ((as_of_ts - df.publish_time) / DAY).clip(lower=0)
    df["decay"] = (-math.log(2) * age / half_life).map(math.exp)
    df["score"] = alpha * df.sim + (1 - alpha) * df.decay if use_decay else df.sim
    df = df.sort_values("score", ascending=False)
    df = df.groupby("bucket").head(k_per_bucket) if use_buckets else df.head(k_total)
    df["date"] = pd.to_datetime(df.publish_time, unit="s").dt.date
    df = df.sort_values("publish_time").reset_index(drop=True)

    if use_asof:                                 # belt and braces
        assert (df.publish_time <= as_of_ts).all(), "LEAK: article after as_of"
    return df

In [12]:
res = retrieve("Nvidia earnings and AI demand", "NVDA", "2023-01-01", "2023-06-30", "2023-05-31")
print(res[["date", "bucket", "sim", "decay", "score", "embed_source", "title"]].to_string())

          date  bucket       sim     decay     score embed_source                                                                       title
0   2023-01-16       0  0.793094  0.043185  0.568121      summary                   3 AI Stocks That Will Make You Amazingly Rich in 10 Years
1   2023-01-18       0  0.775030  0.045227  0.556089      article          Nvidia (NVDA) Dips More Than Broader Markets: What You Should Know
2   2023-01-31       0  0.770657  0.061073  0.557782      article             Nvidia (NVDA) Outpaces Stock Market Gains: What You Should Know
3   2023-02-22       1  0.816595  0.101532  0.602076      article                   Nvidia forecasts first-quarter revenue above expectations
4   2023-02-23       1  0.805961  0.103905  0.595344      article                                  NVIDIA: AI and Gaming Lead The Way Forward
5   2023-02-27       1  0.795579  0.113965  0.591095      article                                    1 Tech Stock Leading the Market Recovery
6   20

## openrouter 


In [3]:
import os
from typing import Literal
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()  

client = OpenAI(base_url="https://openrouter.ai/api/v1",
                api_key=os.environ["OPENROUTER_API_KEY"])      # no key in the code

class RetrievalInput(BaseModel):
    ticker: str = Field(description="Uppercase stock ticker, e.g. NVDA")
    search_query: str = Field(description="Short semantic search phrase, no dates or instructions")
    start: str = Field(description="Start date, YYYY-MM-DD")
    end: str = Field(description="End date, YYYY-MM-DD")
    as_of: str = Field(description="Knowledge cutoff date, YYYY-MM-DD")
    tasks: list[Literal["trend", "event_impact", "point_lookup"]] = Field(
        description="Every task the question asks for. A question can need more than one.")
    event_hint: str = Field(description="Event phrase such as 'May earnings', or empty string if none")

SYSTEM_PROMPT = f"""Extract structured arguments from the user's financial question.
Rules:
- ticker: map company names to tickers. Valid tickers: {', '.join(keep)}.
- Convert month or quarter ranges to exact dates ('Jan to Jun 2023' -> start 2023-01-01, end 2023-06-30).
- If the user says 'as of <date>', set as_of to that date. Otherwise set as_of equal to end.
- tasks: 'trend' for how sentiment or coverage changed over time, 'event_impact' for how the market
  reacted to an event, 'point_lookup' for what was the latest news at a date. Include all that apply.
- event_hint: the event phrase only (for example 'May earnings'); empty string if none.
- search_query: the financial topic only."""

def extract_params(user_text, model="openrouter/free"):
    schema = RetrievalInput.model_json_schema()
    schema["additionalProperties"] = False
    resp = client.chat.completions.create(
        model=model, temperature=0,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_text}],
        response_format={"type": "json_schema",
                         "json_schema": {"name": "retrieval_input", "strict": True, "schema": schema}},
        extra_body={"provider": {"require_parameters": True},
                    "plugins": [{"id": "response-healing"}]},
    )
    return RetrievalInput.model_validate_json(resp.choices[0].message.content)

In [4]:
ticker = "NVDA"
user_msg = (
    "How did sentiment around NVDA change from Jan to Jun 2024, "
    "and how did the market react to its May earnings? Answer as of 31 May 2024."
)

params = extract_params(user_msg)
print("Extracted:", params.model_dump())

Extracted: {'ticker': 'NVDA', 'search_query': 'NVDA sentiment and May earnings market reaction', 'start': '2024-01-01', 'end': '2024-06-30', 'as_of': '2024-05-31', 'tasks': ['trend', 'event_impact'], 'event_hint': 'May earnings'}
